# Parkinson's Drawing Model Comparison

A participant-level comparison of classical and deep-learning image classifiers using the NewHandPD drawing dataset.

## Setup

In [ ]:
import hashlib
import platform
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps
from sklearn.model_selection import StratifiedGroupKFold

PROJECT_ROOT = Path.cwd()

print(f"Python: {platform.python_version()}")
print(f"Project root: {PROJECT_ROOT}")

## Download

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
NEWHANDPD_DIR = DATA_DIR / "raw" / "NewHandPD"
ARCHIVE_DIR = NEWHANDPD_DIR / "archives"
DOWNLOAD_MARKER = NEWHANDPD_DIR / ".download_complete"

ARCHIVES = {
    ("Healthy", "circle"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthyCircle.zip",
    ("Healthy", "meander"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthyMeander.zip",
    ("Healthy", "spiral"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthySpiral.zip",
    ("PD", "circle"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewPatients/PatientCircle.zip",
    ("PD", "meander"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewPatients/PatientMeander.zip",
    ("PD", "spiral"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewPatients/PatientSpiral.zip",
}

download_required = not DOWNLOAD_MARKER.exists()

if download_required:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

    for (label, drawing_type), url in ARCHIVES.items():
        archive_path = ARCHIVE_DIR / f"{label.lower()}_{drawing_type}.zip"
        extraction_dir = NEWHANDPD_DIR / label / drawing_type
        extraction_dir.mkdir(parents=True, exist_ok=True)

        if not archive_path.exists():
            temporary_path = archive_path.with_suffix(".zip.part")
            print(f"Downloading {archive_path.name}...")
            urlretrieve(url, temporary_path)
            temporary_path.replace(archive_path)

        with ZipFile(archive_path) as archive:
            members = [
                member
                for member in archive.infolist()
                if "__MACOSX" not in Path(member.filename).parts
                and not Path(member.filename).name.startswith("._")
            ]
            extraction_root = extraction_dir.resolve()
            member_paths = [
                (extraction_dir / member.filename).resolve()
                for member in members
            ]
            if any(not path.is_relative_to(extraction_root) for path in member_paths):
                raise ValueError(f"Unsafe path found in {archive_path.name}")
            archive.extractall(extraction_dir, members=members)

image_suffixes = {".jpg", ".jpeg", ".png"}
image_paths = sorted(
    path
    for path in NEWHANDPD_DIR.rglob("*")
    if path.suffix.lower() in image_suffixes
    and "__MACOSX" not in path.parts
    and not path.name.startswith("._")
)

if len(image_paths) != 594:
    raise RuntimeError(f"Expected 594 NewHandPD images, found {len(image_paths):,}")

if download_required:
    DOWNLOAD_MARKER.touch()

print(f"Dataset folder: {NEWHANDPD_DIR}")
print(f"Downloaded now: {download_required}")
print(f"Images found: {len(image_paths):,}")
image_paths[:5]

## Inspect data

In [ ]:
if not NEWHANDPD_DIR.is_dir():
    raise FileNotFoundError(f"NewHandPD folder not found: {NEWHANDPD_DIR}")

records = []
invalid_image_paths = []

for image_path in image_paths:
    name_parts = image_path.stem.rsplit("-", maxsplit=1)
    if len(name_parts) != 2:
        invalid_image_paths.append(image_path)
        continue

    task, source_participant_code = name_parts
    participant_number = source_participant_code[1:]

    label = image_path.relative_to(NEWHANDPD_DIR).parts[0]

    if (
        label not in {"Healthy", "PD"}
        or not task[:1].isalpha()
        or not participant_number.isdigit()
    ):
        invalid_image_paths.append(image_path)
        continue

    records.append(
        {
            "path": image_path,
            "label": label,
            "participant_id": f"{label.lower()}-{int(participant_number):02d}",
            "source_participant_code": source_participant_code.upper(),
            "task": task.lower(),
        }
    )

manifest = (
    pd.DataFrame.from_records(records)
    .sort_values(["label", "participant_id", "task"])
    .reset_index(drop=True)
)

class_summary = (
    manifest.groupby("label")
    .agg(images=("path", "size"), participants=("participant_id", "nunique"))
    .sort_index()
)
task_summary = (
    manifest.groupby(["task", "label"])
    .size()
    .unstack(fill_value=0)
)

print(f"NewHandPD images: {len(manifest):,}")
print(f"Invalid image filenames: {len(invalid_image_paths):,}")
display(class_summary)
display(task_summary)
manifest.head()

## Validate images

In [ ]:
image_metadata_records = []
invalid_image_records = []

for row in manifest.itertuples(index=False):
    try:
        with Image.open(row.path) as image:
            image_format = image.format
            image_mode = image.mode
            width, height = image.size
            image.verify()

        with row.path.open("rb") as image_file:
            sha256 = hashlib.file_digest(image_file, "sha256").hexdigest()

        image_metadata_records.append(
            {
                "path": row.path,
                "format": image_format,
                "mode": image_mode,
                "width": width,
                "height": height,
                "sha256": sha256,
            }
        )
    except (OSError, SyntaxError) as error:
        invalid_image_records.append(
            {"path": row.path, "error": str(error)}
        )

invalid_images = pd.DataFrame(
    invalid_image_records,
    columns=["path", "error"],
)

if not invalid_images.empty:
    display(invalid_images)
    raise RuntimeError(f"Image validation failed for {len(invalid_images)} files")

image_metadata = pd.DataFrame.from_records(image_metadata_records)
validated_manifest = manifest.merge(
    image_metadata,
    on="path",
    how="left",
    validate="one_to_one",
)

image_profiles = (
    validated_manifest.groupby(["format", "mode", "width", "height"])
    .size()
    .rename("images")
    .reset_index()
    .sort_values("images", ascending=False)
)

duplicate_images = validated_manifest[
    validated_manifest.duplicated("sha256", keep=False)
].sort_values(["sha256", "label", "participant_id", "task"])
duplicate_summary = (
    duplicate_images.groupby("sha256")
    .agg(
        files=("path", "size"),
        participants=("participant_id", "nunique"),
        labels=("label", "nunique"),
        tasks=("task", "nunique"),
    )
    .reset_index()
)

expected_tasks = {"circa", "mea1", "mea2", "mea3", "mea4", "sp1", "sp2", "sp3", "sp4"}
participant_task_sets = validated_manifest.groupby("participant_id")["task"].agg(set)
task_issue_records = []

for participant_id, observed_tasks in participant_task_sets.items():
    missing_tasks = sorted(expected_tasks - observed_tasks)
    unexpected_tasks = sorted(observed_tasks - expected_tasks)
    if missing_tasks or unexpected_tasks:
        task_issue_records.append(
            {
                "participant_id": participant_id,
                "missing_tasks": missing_tasks,
                "unexpected_tasks": unexpected_tasks,
            }
        )

task_issues = pd.DataFrame(
    task_issue_records,
    columns=["participant_id", "missing_tasks", "unexpected_tasks"],
)

expected_prefix = {"Healthy": "H", "PD": "P"}
participant_code_issues = validated_manifest[
    validated_manifest.apply(
        lambda row: row.source_participant_code[:1] != expected_prefix[row.label],
        axis=1,
    )
]["path label participant_id source_participant_code task".split()]

print(f"Images validated: {len(validated_manifest):,}")
print(f"Unreadable images: {len(invalid_images):,}")
print(f"Exact duplicate groups: {len(duplicate_summary):,}")
print(f"Duplicate groups spanning participants: {(duplicate_summary['participants'] > 1).sum():,}")
print(f"Duplicate groups spanning labels: {(duplicate_summary['labels'] > 1).sum():,}")
print(f"Participants with task issues: {len(task_issues):,}")
print(f"Participant-code prefix issues: {len(participant_code_issues):,}")
display(image_profiles.head(10))
display(task_issues)
participant_code_issues.head()

## Clean manifest

The source file `mea5-P8.jpg` is retained unchanged and its filename-derived task remains available as `source_task`. Because participant P8 has `mea1`, `mea2`, `mea3`, and `mea5` but no `mea4`, while the dataset structure contains four meander repetitions, the analytical `task` value is normalised from `mea5` to `mea4`. This is an explicit inference from the dataset structure, not an author-confirmed correction.

In [ ]:
clean_manifest = validated_manifest.copy()
clean_manifest = clean_manifest.rename(columns={"task": "source_task"})
clean_manifest["task"] = clean_manifest["source_task"]

task_normalisations = {
    ("pd-08", "mea5"): "mea4",
}

for (participant_id, source_task), normalised_task in task_normalisations.items():
    normalisation_mask = (
        clean_manifest["participant_id"].eq(participant_id)
        & clean_manifest["source_task"].eq(source_task)
    )
    if normalisation_mask.sum() != 1:
        raise RuntimeError(
            f"Expected one {source_task} image for {participant_id}, "
            f"found {normalisation_mask.sum()}"
        )
    clean_manifest.loc[normalisation_mask, "task"] = normalised_task

participant_numbers = (
    clean_manifest["participant_id"].str.rsplit("-", n=1).str[-1].astype(int)
)
participant_prefixes = clean_manifest["label"].map({"Healthy": "H", "PD": "P"})
clean_manifest["participant_code"] = participant_prefixes + participant_numbers.astype(str)
clean_manifest["participant_code_normalised"] = (
    clean_manifest["participant_code"] != clean_manifest["source_participant_code"]
)
clean_manifest["task_normalised"] = (
    clean_manifest["task"] != clean_manifest["source_task"]
)
clean_manifest["is_exact_duplicate"] = clean_manifest.duplicated(
    "sha256",
    keep=False,
)

parent = {
    participant_id: participant_id
    for participant_id in clean_manifest["participant_id"].unique()
}

def find_group(participant_id):
    while parent[participant_id] != participant_id:
        parent[participant_id] = parent[parent[participant_id]]
        participant_id = parent[participant_id]
    return participant_id

def join_groups(first_participant, second_participant):
    first_root = find_group(first_participant)
    second_root = find_group(second_participant)
    if first_root != second_root:
        parent[second_root] = first_root

duplicate_rows = clean_manifest[clean_manifest["is_exact_duplicate"]]
for _, duplicate_group in duplicate_rows.groupby("sha256"):
    participants = sorted(duplicate_group["participant_id"].unique())
    for participant_id in participants[1:]:
        join_groups(participants[0], participant_id)

clean_manifest["split_group"] = clean_manifest["participant_id"].map(find_group)

normalised_task_sets = clean_manifest.groupby("participant_id")["task"].agg(set)
if not normalised_task_sets.map(lambda tasks: tasks == expected_tasks).all():
    raise RuntimeError("Task normalisation did not produce nine expected tasks per participant")

duplicate_split_counts = duplicate_rows.assign(
    split_group=duplicate_rows["participant_id"].map(find_group)
).groupby("sha256")["split_group"].nunique()
if not duplicate_split_counts.eq(1).all():
    raise RuntimeError("An exact duplicate group spans multiple split groups")

print(f"Task labels normalised under the documented rule: {clean_manifest['task_normalised'].sum():,}")
print(f"Participant codes normalised from class and participant number: {clean_manifest['participant_code_normalised'].sum():,}")
print(f"Images in exact duplicate groups: {clean_manifest['is_exact_duplicate'].sum():,}")
print(f"Participants: {clean_manifest['participant_id'].nunique():,}")
print(f"Leakage-safe split groups: {clean_manifest['split_group'].nunique():,}")
clean_manifest[
    [
        "path",
        "label",
        "participant_id",
        "participant_code",
        "task",
        "split_group",
        "is_exact_duplicate",
    ]
].head()

## Preview images

Display all nine drawings for participant 8 in each class. The preview uses the analytical task labels while preserving any different filename-derived label in the caption.

In [ ]:
preview_participants = {
    "Healthy": "healthy-08",
    "PD": "pd-08",
}
preview_tasks = ["circa", "mea1", "mea2", "mea3", "mea4", "sp1", "sp2", "sp3", "sp4"]
thumbnail_size = (180, 180)
caption_height = 24
columns = 3
rows = 3
gap = 10

for label, participant_id in preview_participants.items():
    participant_images = (
        clean_manifest.loc[
            clean_manifest["label"].eq(label)
            & clean_manifest["participant_id"].eq(participant_id)
        ]
        .set_index("task")
        .loc[preview_tasks]
    )

    canvas_width = columns * thumbnail_size[0] + (columns + 1) * gap
    canvas_height = rows * (caption_height + thumbnail_size[1]) + (rows + 1) * gap
    canvas = Image.new("RGB", (canvas_width, canvas_height), "white")
    draw = ImageDraw.Draw(canvas)

    for index, (task, image_row) in enumerate(participant_images.iterrows()):
        with Image.open(image_row["path"]) as source_image:
            thumbnail = ImageOps.contain(source_image.convert("RGB"), thumbnail_size)

        column = index % columns
        grid_row = index // columns
        x = gap + column * (thumbnail_size[0] + gap)
        y = gap + grid_row * (caption_height + thumbnail_size[1] + gap)
        image_x = x + (thumbnail_size[0] - thumbnail.width) // 2
        image_y = y + caption_height + (thumbnail_size[1] - thumbnail.height) // 2
        canvas.paste(thumbnail, (image_x, image_y))

        source_task = image_row["source_task"]
        caption = task if source_task == task else f"{task} (source: {source_task})"
        draw.text((x, y), caption, fill="black")

    participant_code = participant_images.iloc[0]["participant_code"]
    print(f"{label}: {participant_id} ({participant_code})")
    display(canvas)

## Split data

Reserve one fold as a fixed test set before preprocessing or modelling. Five-fold stratified group splitting gives an approximately 80/20 division while keeping each `split_group` wholly within one partition. The exact proportion can differ because duplicate-linked participant groups are indivisible.

In [ ]:
random_state = 42
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)

train_indices, test_indices = next(
    splitter.split(
        clean_manifest,
        y=clean_manifest["label"],
        groups=clean_manifest["split_group"],
    )
)

split_manifest = clean_manifest.copy()
split_manifest["partition"] = "train"
split_manifest.loc[split_manifest.index[test_indices], "partition"] = "test"

participant_partition_counts = (
    split_manifest.groupby("participant_id")["partition"].nunique()
)
if not participant_partition_counts.eq(1).all():
    raise RuntimeError("A participant appears in both train and test partitions")

group_partition_counts = split_manifest.groupby("split_group")["partition"].nunique()
if not group_partition_counts.eq(1).all():
    raise RuntimeError("A leakage-safe split group appears in both partitions")

duplicate_partition_counts = (
    split_manifest.loc[split_manifest["is_exact_duplicate"]]
    .groupby("sha256")["partition"]
    .nunique()
)
if not duplicate_partition_counts.eq(1).all():
    raise RuntimeError("Exact duplicate content appears in both partitions")

labels_per_partition = split_manifest.groupby("partition")["label"].nunique()
if not labels_per_partition.eq(clean_manifest["label"].nunique()).all():
    raise RuntimeError("A partition does not contain both diagnostic classes")

split_summary = (
    split_manifest.groupby(["partition", "label"])
    .agg(
        images=("path", "size"),
        participants=("participant_id", "nunique"),
        split_groups=("split_group", "nunique"),
    )
    .sort_index()
)

print(f"Random state: {random_state}")
print("Participant overlap: 0")
print("Split-group overlap: 0")
print("Exact-duplicate overlap: 0")
display(split_summary)

## Preprocess images

Apply the same minimal preprocessing to every image: correct any stored orientation, convert to greyscale, resize to fit within `128 × 128` pixels without changing the aspect ratio, centre on a white square, and scale pixel intensities to `[0, 1]`. Processing is performed in memory; the source files remain unchanged and no resized copies are written to disk.

In [ ]:
image_size = (128, 128)

def preprocess_image(image_path, output_size=image_size):
    with Image.open(image_path) as source_image:
        greyscale_image = ImageOps.exif_transpose(source_image).convert("L")
        resized_image = ImageOps.contain(
            greyscale_image,
            output_size,
            method=Image.Resampling.LANCZOS,
        )

    processed_image = Image.new("L", output_size, color=255)
    offset = (
        (output_size[0] - resized_image.width) // 2,
        (output_size[1] - resized_image.height) // 2,
    )
    processed_image.paste(resized_image, offset)
    return np.asarray(processed_image, dtype=np.float32) / 255.0

preprocessed_images = np.stack(
    [preprocess_image(image_path) for image_path in split_manifest["path"]],
)
split_manifest = split_manifest.copy()
split_manifest["image_index"] = np.arange(len(split_manifest))

expected_shape = (len(split_manifest), image_size[1], image_size[0])
if preprocessed_images.shape != expected_shape:
    raise RuntimeError(
        f"Expected preprocessed shape {expected_shape}, found {preprocessed_images.shape}"
    )
if not np.isfinite(preprocessed_images).all():
    raise RuntimeError("Preprocessed images contain non-finite values")
if preprocessed_images.min() < 0.0 or preprocessed_images.max() > 1.0:
    raise RuntimeError("Preprocessed pixel values fall outside [0, 1]")

preprocessing_summary = pd.Series(
    {
        "images": len(preprocessed_images),
        "array shape": str(preprocessed_images.shape),
        "data type": str(preprocessed_images.dtype),
        "minimum pixel value": float(preprocessed_images.min()),
        "maximum pixel value": float(preprocessed_images.max()),
        "memory (MiB)": preprocessed_images.nbytes / (1024 ** 2),
    },
    name="value",
)
display(preprocessing_summary)